### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="porto_seguro",
    dataset_year="2017",
    domain_str="insurance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/c/porto-seguro-safe-driver-prediction",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c porto-seguro-safe-driver-prediction -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/porto_seguro && mv train.csv local-data-warehouse/porto_seguro/
""",
    # References
    academic_reference_bibtex=r"""@misc{Howard2017PortoSegurosSafeDriverPrediction,
  author = {Addison Howard and Adriano Moala and Walter Reade},
  title  = {Porto Seguro’s Safe Driver Prediction},
  year   = {2017},
  howpublished = {\url{https://kaggle.com/competitions/porto-seguro-safe-driver-prediction}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Howard2017PortoSegurosSafeDriverPrediction",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized, so feature meanings are unknown.
- Following the expert solutions on Kaggle, we drop *calc features.
- In general, the 2nd and 3rd place did a lot of interesting feature engineering for interactions. We do not copy these as the data is clean as it is.
- We drop the index as it seems to be uninformative for this dataset.
- There exist only 6 duplicates in the training data. We drop them as the portion is too small to be meaningful.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    objective_metric_name="normalized_gini_coefficient", # see https://www.kaggle.com/c/ClaimPredictionChallenge/discussion/703
    stratify_on="target",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

# Create real nan values
df = df.replace(-1, np.nan)

# Drop *calc features
calc_features = [col for col in df.columns if "calc" in col]
df = df.drop(columns=calc_features)

# Drop index
df = df.drop(columns=["id"])

# Dtypes
as_cat_type = [
    col for col in df.columns if col.endswith("cat") or col.endswith("bin")
]
df[as_cat_type] = df[as_cat_type].astype("category")
df[task_mold.target_column_name] = df[task_mold.target_column_name].astype("category")

# Drop duplicates w/o target
df = df.drop_duplicates(subset=[col for col in df.columns if col != task_mold.target_column_name])
df = df.reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()